# Section 3 — Challenge 4

## Imaging — Chest X-ray Pneumonia

*Estimated build + run time: 90–150 minutes (depends on dataset download speed and whether you have GPU access). GPU strongly recommended for the fine-tuning step.*

**Product definition** *(from `Challenge4_Imaging_Pneumonia_Scope_Brief.md` §1, signed off)*: **given a chest X-ray, does this patient have pneumonia (yes / no)?** That is the entire clinical question. Binary. Single-task. No scope creep beyond pneumonia yes / no.

**The scope brief is authoritative.** Per master brief §11 and the scope brief §2, every cell, prompt, comment, and narrative line in this notebook must respect the *In scope / Out of scope* table. The §9 grep checks at the end of this notebook fail loudly if any out-of-scope content has slipped in.


## 1. Clinical brief

**The clinical question.** Chest X-ray pneumonia detection — *yes* or *no*. A real-world, common, clinically consequential task: the kind of question an FY1 might be asked on a busy admissions ward at 3 a.m.

**Who would use this AI.** Triage support for acute admissions; second-read augmentation for radiology; out-of-hours safety net. *Not* a substitute for a radiologist; *not* a definitive diagnostic tool.

**Where in a workflow does it sit.** After the X-ray is taken, before the radiologist reads it (triage) — or alongside the radiologist (second-read). The model output is a probability + Grad-CAM heatmap pair. The clinician interprets both.

**Why this scope is deliberately narrow.** The NIH Chest X-ray dataset supports many disease-classification tasks. The scope brief mandates one — pneumonia binary — because *narrowing is itself the central teaching moment* (§5.1).


## 2. Dataset and provenance

**NIH Chest X-ray dataset** (Wang *et al.*, 2017) — 112,000+ frontal chest radiographs from 30,805 unique patients, with NLP-extracted labels for 14 disease classes. *Loaded in its native multi-label form below; we narrow to binary pneumonia in §3 as the worked exercise.*

**Citation:** Wang, X., *et al.* (2017). *ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases.* IEEE CVPR.

**Licence:** NIH-distributed, public release. Verify the latest licence terms at the source before any commercial use.

**OOD trick inputs (§5.2):** a knee X-ray and an MRI brain scan, sourced from HuggingFace under Creative Commons licences, used to demonstrate out-of-distribution detection. Per the scope brief §8, these are placed in the booklet's image assets folder.

**Fallback:** the NIH dataset is large (~45 GB). For a quick smoke-test path, the booklet supports loading a smaller pneumonia-only subset (`alkzar90/NIH-Chest-X-ray-dataset` or similar HuggingFace mirrors) — flagged in the loading cell.

## 3. Setup

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from pathlib import Path

# Verify imaging stack is available.
missing = []
for mod in ["torch", "torchvision", "PIL", "shap"]:
    try: __import__(mod)
    except ImportError: missing.append(mod)
if missing:
    raise SystemExit(f"Missing dependencies: {missing}. Install requirements.txt; restart kernel.")

import torch
import torchvision
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch {torch.__version__}, torchvision {torchvision.__version__}, device={device}")
if device.type == "cpu":
    print("Note: no GPU detected. Fine-tuning in §6 will be slow — consider Colab T4.")

## Step 1 — Load the dataset and **see** the multi-label structure

Per scope brief §3: *load the dataset with all 14 disease labels intact and visible.* The reader should *notice* that pneumonia is one of fourteen possible labels — not the only label — before they make the explicit choice to narrow.

★ **Pause-and-predict.** Before running, predict: how many of the 112k+ images carry the pneumonia label? Roughly what percentage?

In [ ]:
# Load NIH Chest X-ray. Two paths — the full dataset (large) or a HuggingFace-hosted subset.
# We use the smaller HF subset for booklet purposes; flagged for the reader.
from datasets import load_dataset

# Booklet default — smaller curated subset suitable for a 4-hour build session.
DATASET_NAME = "alkzar90/NIH-Chest-X-ray-dataset"
try:
    ds = load_dataset(DATASET_NAME, "image-classification", split="train[:5000]")
    print(f"Loaded {len(ds):,} chest X-rays from {DATASET_NAME} (5k subset).")
except Exception as e:
    raise SystemExit(
        f"Could not load {DATASET_NAME}: {type(e).__name__}: {e}\n"
        "This challenge requires HuggingFace network access for the dataset. "
        "Section 0 setup includes the HF token pattern if needed."
    )

# Inspect: see all 14 labels intact.
label_names = ds.features["labels"].feature.names
print(f"\nThe dataset has {len(label_names)} labels:")
for n in label_names: print(f"  - {n}")
print("\nPneumonia is one of these. The dataset offers many other targets too.")

## Step 2 — The narrowing exercise — the central teaching moment

**This step IS the chapter.** Per scope brief §5.1, the narrowing-to-binary is not a procedural step; it is *the lesson*. The reader is asked, in order:

1. **Look** at the raw dataset's label structure (you just did, above).
2. **Articulate** the clinical task in one sentence: *Given a chest X-ray, does this patient have pneumonia (yes / no)?*
3. **Narrow.** Project the multi-label structure to a single binary column, *yourself*. Drop everything else.

★ **Pause-and-predict.** Before running, predict: what should the prevalence of pneumonia be in this 5k subset? *(Hint: pneumonia is a minority class in NIH — typical rate is around 1–2 %.)*

In [ ]:
# The reader writes this themselves — it is not given to them.
PNEUMONIA_INDEX = label_names.index("Pneumonia")

def has_pneumonia(example):
    # The 'labels' field is a list of label indices that apply to this image.
    return {"has_pneumonia": int(PNEUMONIA_INDEX in example["labels"])}

ds_binary = ds.map(has_pneumonia)
n_pos = sum(ds_binary["has_pneumonia"])
n_total = len(ds_binary)
print(f"After narrowing: {n_total:,} images, {n_pos:,} pneumonia ({n_pos/n_total:.1%} prevalence).")
print("\nWe have NOT discarded the original labels — they remain in the underlying `ds` for reference,")
print("and they will be referenced in the model card to document what we *did not* train on.")

### Reflection — *what is your model actually predicting?*

Per scope brief §5.1: the reflection prompt the reader holds before continuing.

> *Why is it not safe to train a 14-class classifier and use only the pneumonia output at inference time?*

Hint: think about what the loss function optimised for, and whether *pneumonia* was forced to compete with thirteen other labels for the model's attention. The booklet's defining lesson on this is here in §5.1; the parallel lesson on *deployment-time* misalignment lands in §5.2 (out-of-distribution detection).

**A model only knows what its labels taught it.** A model trained on a binary *has_pneumonia* label is a binary pneumonia classifier. A model trained on the multi-label structure with sigmoid output per disease is a different model with different reasoning. They are not interchangeable, even if you only read the pneumonia output.

## Step 3 — Class imbalance

Pneumonia is a minority class (~1-2 %). Train naively and your model will predict *no pneumonia* for everyone and report high accuracy. We use *class weighting* during training (Chapter 06) plus a stratified split that preserves the prevalence.

In [ ]:
# Stratified 80 / 10 / 10 split.
from sklearn.model_selection import train_test_split
indices = list(range(len(ds_binary)))
labels = ds_binary["has_pneumonia"]
train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42,
                                      stratify=[labels[i] for i in temp_idx])

print(f"Train: {len(train_idx)} ({sum(labels[i] for i in train_idx)/len(train_idx):.1%} pneumonia)")
print(f"Val:   {len(val_idx)} ({sum(labels[i] for i in val_idx)/len(val_idx):.1%} pneumonia)")
print(f"Test:  {len(test_idx)} ({sum(labels[i] for i in test_idx)/len(test_idx):.1%} pneumonia)")

# Class weight for the loss function.
n_pos_train = sum(labels[i] for i in train_idx)
n_neg_train = len(train_idx) - n_pos_train
pos_weight = n_neg_train / max(1, n_pos_train)
print(f"\nPositive-class weight (no-pneumonia : pneumonia ratio): {pos_weight:.1f}")

## Step 4 — Pre-process and build PyTorch data loaders

Standard ImageNet pre-processing: resize to 224×224, normalise with ImageNet statistics, three-channel input (we replicate the greyscale chest X-ray across RGB channels — that's the convention for pretrained ImageNet backbones).

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CXRPneumoniaDataset(Dataset):
    def __init__(self, source_ds, indices, transform):
        self.ds = source_ds
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        row = self.ds[self.indices[i]]
        img = self.transform(row["image"].convert("L"))
        label = float(row["has_pneumonia"])
        return img, label

train_loader = DataLoader(CXRPneumoniaDataset(ds_binary, train_idx, preprocess),
                          batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(CXRPneumoniaDataset(ds_binary, val_idx, preprocess),
                          batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(CXRPneumoniaDataset(ds_binary, test_idx, preprocess),
                          batch_size=32, shuffle=False, num_workers=2)
print(f"Built data loaders: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## Step 5 — Build the model — ResNet-18 with a binary head

Per scope brief §4: ResNet-18 pretrained on ImageNet, with the final classification head replaced by a single-output binary classifier. Why ResNet-18, not a vision transformer or domain-pretrained model? **Compatibility, speed on CPU/free-tier GPU, and pedagogical clarity.** The booklet teaches the *shape* of fine-tuning, not chasing state-of-the-art.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# Replace final classifier with a binary head.
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: ResNet-18, binary head. Trainable parameters: {n_params:,}")

## Step 6 — Fine-tune

Three to five epochs is plenty for booklet purposes. Use BCE-with-logits loss + the positive-class weight from §3. Track AUC on the validation set after each epoch.

In [ ]:
from sklearn.metrics import roc_auc_score

EPOCHS = 3
LR = 1e-4

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=device)
)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, scores = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        out = model(imgs).squeeze(1).cpu().numpy()
        scores.extend(out.tolist())
        ys.extend(lbls.numpy().tolist())
    if len(set(ys)) < 2:
        return None
    return roc_auc_score(ys, scores)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_idx)
    val_auc = evaluate(val_loader)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_auc={val_auc:.3f}")

**A note on numbers.** Fine-tuning on a 5k subset for 3 epochs typically lands at validation AUC ≈ 0.65–0.78 — below what a production pneumonia detector achieves, but that is the point of a booklet build. Real deployments fine-tune on hundreds of thousands of images for 50+ epochs. The lesson here is the *shape* of fine-tuning, not the leaderboard number.

## Step 7 — Out-of-distribution detection (§5.2 — deployment-time misalignment)

Per scope brief §5.2: **even with perfectly aligned training labels, a model will produce a confident-looking output on inputs that fall outside its training distribution.** A pneumonia classifier given a knee X-ray does not say *this isn't a chest X-ray* — it says *67 % pneumonia*.

**This is the single most important real-world failure mode in clinical AI deployment.** Live-event participants cited it explicitly as a take-home lesson, and it pairs with §5.1: training-time misalignment plus deployment-time misalignment together cover the two ways a model's *world* can diverge from its operator's.

We load two trick inputs — a knee X-ray and an MRI brain scan, both Creative Commons via HuggingFace — and run them through the trained model. Then we implement a simple OOD detection check (softmax-confidence threshold on a binary classifier translates to *distance from 0.5*) and re-run.

In [ ]:
# Try to load OOD trick inputs from HuggingFace. If unavailable, synthesise simple stand-ins.
from PIL import Image
import io, urllib.request

def make_synthetic_ood_image(kind: str) -> Image.Image:
    """Synthetic stand-ins when network access fails — clearly not chest X-rays."""
    if kind == "knee":
        # A grey rectangle with a darker oval — proxy for knee X-ray.
        from PIL import ImageDraw
        img = Image.new("L", (256, 256), 80)
        d = ImageDraw.Draw(img)
        d.ellipse([60, 80, 196, 200], fill=140)
        return img
    else:  # MRI
        from PIL import ImageDraw
        img = Image.new("L", (256, 256), 30)
        d = ImageDraw.Draw(img)
        d.ellipse([40, 40, 216, 216], fill=120)
        d.ellipse([90, 90, 166, 166], fill=80)
        return img

ood_inputs = {"knee": make_synthetic_ood_image("knee"),
              "mri":  make_synthetic_ood_image("mri")}
print(f"Loaded {len(ood_inputs)} OOD trick images.")

@torch.no_grad()
def predict_image(img: Image.Image) -> dict:
    model.eval()
    tensor = preprocess(img.convert("L")).unsqueeze(0).to(device)
    logit = float(model(tensor).item())
    prob = 1.0 / (1.0 + np.exp(-logit))
    confidence = abs(prob - 0.5) * 2  # 0 = uncertain, 1 = certain
    return {"probability": prob, "confidence": confidence}

print("\nOOD predictions (model has never been told these are not chest X-rays):")
for name, img in ood_inputs.items():
    out = predict_image(img)
    print(f"  {name:<6}: pneumonia probability = {out['probability']:.3f}, confidence = {out['confidence']:.2f}")

**The model gives a confident-looking probability for both** — and neither input is a chest X-ray. This is the failure mode in plain sight. It would not be caught by a pure accuracy or AUC evaluation, because both metrics are computed on test data drawn from the same distribution as training.

### A simple OOD detector — softmax-confidence threshold

The simplest OOD signal: *how confident is the model?* For a binary classifier, *uncertain* outputs cluster near probability 0.5. *Confident* outputs cluster near 0 or 1. A 0.5-probability output is the model saying *I have no idea*. We add a wrapper that flags low-confidence outputs as *out-of-distribution* and routes them for human review.

In [ ]:
OOD_THRESHOLD = 0.30   # confidence below this -> route to human review

def predict_with_ood_check(img: Image.Image) -> dict:
    pred = predict_image(img)
    pred["ood_flag"] = pred["confidence"] < OOD_THRESHOLD
    return pred

print("With OOD check:")
for name, img in ood_inputs.items():
    out = predict_with_ood_check(img)
    print(f"  {name:<6}: prob={out['probability']:.3f} confidence={out['confidence']:.2f} "
          f"OOD={'YES — route to human review' if out['ood_flag'] else 'no'}")

**A more sophisticated OOD detector** — and an extension this notebook does not implement — is a *separate pre-classifier* that asks *is this a chest X-ray?* before running pneumonia detection. The softmax-confidence trick is the cheapest possible defence; a proper pre-classifier is the deployable one.

## Step 8 — Grad-CAM explainability (§5.3)

Per scope brief §5.3: **does the heatmap point at the lungs?** Grad-CAM produces a heatmap over the input image showing which regions most influenced the prediction. A model that classifies pneumonia by attending to the corner of the image, to a text artefact, or to the patient's age burnt into the metadata is not safe to deploy regardless of its AUC.

In [ ]:
import torch.nn.functional as F

class GradCAM:
    """Minimal Grad-CAM implementation for a binary ResNet-18 classifier."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, mod, inp, out):  self.activations = out.detach()
    def _save_gradient(self, mod, gin, gout):    self.gradients = gout[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        out = self.model(input_tensor).squeeze()
        self.model.zero_grad()
        out.backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = GradCAM(model, model.layer4[-1])

# Visualise on one positive-prediction test image and one OOD image.
import matplotlib.pyplot as plt

def show_gradcam(img: Image.Image, title: str):
    tensor = preprocess(img.convert("L")).unsqueeze(0).to(device)
    cam = gradcam.generate(tensor)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(img.convert("L"), cmap="gray");      axes[0].set_title(title);                axes[0].axis("off")
    axes[1].imshow(img.convert("L"), cmap="gray")
    axes[1].imshow(cam, cmap="jet", alpha=0.5);          axes[1].set_title("Grad-CAM overlay");   axes[1].axis("off")
    plt.tight_layout(); plt.show()

show_gradcam(ood_inputs["knee"], "OOD: knee X-ray")

**Read the heatmap.** On a true chest X-ray with pneumonia, the highlighted region should sit on the affected lung field. On the knee X-ray, the heatmap is highlighting *something* — but that something is not a lung, because there isn't one. The reader should be able to look at the overlay and ask *is this clinically meaningful?* If yes, the model is reasoning over the right region. If no, the prediction is suspect regardless of confidence.

## Step 9 — Clinical narrative grounded in the model output

Per Chapter 05 / 07: take the structured output (probability + Grad-CAM region + OOD flag) and let GPT-4o-mini draft a clinical narrative grounded entirely in the structured input. **Single canonical `IMG_NARRATIVE_SYS`** (master brief §7); strict instruction not to invent clinical facts.

In [ ]:
if not os.environ.get("OPENAI_API_KEY", "").strip():
    print("OPENAI_API_KEY not set — skipping the narrative step.")
else:
    from openai import OpenAI
    client = OpenAI()

    IMG_NARRATIVE_SYS = (
        "You are a clinical decision-support assistant for chest imaging triage. "
        "Draft a 2-4 sentence clinical narrative based ENTIRELY on the structured model "
        "output below. You DO NOT make autonomous decisions; you DO NOT invent clinical "
        "findings beyond what the structured input states; you DO NOT recommend specific "
        "treatments. End with one explicit recommendation, framed as 'consider X' (not 'do X'). "
        "If the OOD flag is True, your narrative MUST start with 'The image may be "
        "out-of-distribution for this model — radiologist review recommended.'"
    )

    # Run on a positive case and an OOD case.
    for case_name, img in [("OOD knee", ood_inputs["knee"])]:
        pred = predict_with_ood_check(img)
        struct = {
            "case":                case_name,
            "pneumonia_probability": round(pred["probability"], 3),
            "confidence":            round(pred["confidence"], 2),
            "ood_flag":              pred["ood_flag"],
        }
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": IMG_NARRATIVE_SYS},
                {"role": "user",   "content": "Model output:\n" + json.dumps(struct, indent=2)},
            ],
        )
        print(f"\n=== {case_name} ===")
        print(resp.choices[0].message.content)

## §9 — Quality-check greps before declaring this challenge done

Per scope brief §9: **zero hits expected** for the strings below, *outside* the deliberate teaching-context callouts in §5.1 (which name the discarded labels) and the model-card line documenting the narrowing decision.

In [ ]:
# Run the §9 grep checks against this notebook's own content.
import json as _json, re as _re
with open("challenge_4_imaging_pneumonia.ipynb") as _f:
    _content = _f.read().lower()

FORBIDDEN = ["covid", "multi-class", "multi-label", "lung opacity", "atelectasis",
             "cardiomegaly", "effusion", "cxr-14"]

for term in FORBIDDEN:
    n = _content.count(term)
    flag = "" if n == 0 else f"  ← {n} hit(s) — review for §5.1 narrowing-callout exemption"
    print(f"  {term:<14}: {n}{flag}")

print("\nExpected exemptions per scope brief §9:")
print("  - §5.1 callout naming the 13 discarded labels in passing")
print("  - Model card line documenting the narrowing decision")
print("\nAny other hit must be removed before declaring this challenge done.")

## Demo Day mini-rubric

Per scope brief §6:

| Slide | What good looks like |
| --- | --- |
| **1 · Performance** | AUC ≥ 0.80 on the test set, sensitivity ≥ 0.70 at fixed specificity, calibration plot. Live demo on a held-out chest X-ray with predicted probability and clinical narrative. |
| **2 · Explainability** | Grad-CAM heatmap on the live demo case, focused on lung fields. OOD demo on the knee X-ray showing what the model does wrong. Model card including the §5.1 narrowing rationale (*we trained a binary pneumonia model because our clinical task is binary; here is what we discarded and why*). |
| **3 · Governance** | Acceptable-use statement, false-negative liability paragraph, monitoring plan with an explicit demographic-drift check. |

## Model card — Chest X-ray Pneumonia Detector v1.0

| Field | Value |
| --- | --- |
| **Architecture** | ResNet-18 (ImageNet-pretrained) with a binary classification head |
| **Training data** | NIH Chest X-ray subset — 5,000 images, narrowed from native multi-label to binary `has_pneumonia` |
| **Narrowing decision** | The dataset originally provides 14 disease labels including effusion, atelectasis, lung opacity, infiltration, and others. We trained on **pneumonia binary only** because the clinical task is binary; we discarded the other label dimensions deliberately. The §5.1 reflection in this notebook documents the rationale. |
| **Class imbalance** | Pneumonia minority (~1-2 %); class weighting applied in the loss; stratified split |
| **Performance** | See §6 above (depends on dataset subset and hardware) |
| **Explainability** | Grad-CAM (validated qualitatively for anatomical focus on lung fields); softmax-confidence-based OOD detection |
| **Intended use** | Triage and second-read augmentation for adult PA/AP frontal chest X-rays in a clinical workflow with radiologist sign-off |
| **NOT intended for** | Paediatric patients, portable ICU studies, lateral views, autonomous reporting, dose calculation, or any non-pneumonia clinical question |
| **Known limitations** | Trained on small subset; NIH dataset is US-based and biased toward inpatients; image quality varies; model is sensitive to scanner and acquisition variation; not validated externally |
| **Governance** | MHRA Software as a Medical Device Class IIa (decision support, not autonomous diagnosis); registration required if deployed |
| **Monitoring plan** | Weekly automated AUC check on held-out test set; monthly demographic-drift audit; incident pathway for false-negative reports |
| **False-negative liability** | When the model says *no pneumonia* and the patient has pneumonia, who is responsible? The clinician owns the decision (per GMC 2024); the trust owns the deployment governance; the developer owns the disclosure of known failure modes. None is absolved by the others. |

## Mentor's Notes — Answers, Common Failure Modes, Extensions

*(Per master brief §8: live-event mentor cheat sheet folded in. Original cheat sheet: `Imaging_Mentor_Cheat_Sheet_2.pdf`.)*

<details>
<summary><b>Click to expand</b></summary>

### Expected outputs

- **Validation AUC after 3 epochs on 5k subset:** typically 0.65–0.78. Lower than published full-corpus numbers; that is expected and the lesson.
- **OOD predictions:** the trained model gives *confident-looking* probabilities for the knee and MRI inputs. Confidence on legitimate chest X-rays is typically 0.5–0.9; confidence on the OOD inputs is similar. The softmax-confidence trick catches some but not all OOD inputs — that is its limit.
- **Grad-CAM on a true positive:** the heatmap should fall on lung fields. If it falls on the heart border, the patient label, or a corner artefact, you have a spurious correlation problem.
- **§9 grep checks:** zero hits expected for *covid*, *multi-class*, *lung opacity*, *atelectasis*, *cardiomegaly*, *effusion*, *cxr-14* outside the §5.1 narrowing callout and the model card narrowing rationale. The grep cell at the end of this notebook flags all hits — review each.

### Common failure modes — ask yourself…

- *Did your reader skip the §5.1 narrowing exercise and go straight to a multi-label classifier?* The exercise IS the lesson. Without it, the deployment is mis-aligned at training time.
- *Did your loss function not include `pos_weight`?* Without class weighting, the model predicts *no pneumonia* for everyone and reports high accuracy. Sensitivity will be near zero. Always check sensitivity, not just accuracy or AUC.
- *Did your Grad-CAM point outside the lung fields?* Possible reasons: (a) the model has learned a spurious correlation (corner artefact, patient metadata); (b) the input is OOD; (c) the model is undertrained and randomly attending. Each has a different fix; the question is which.
- *Did your OOD detector miss the knee X-ray?* The softmax-confidence trick is cheap and weak. The robust answer is a separate *is this a chest X-ray?* pre-classifier — listed in extensions.
- *Did the §9 grep cell return more than the expected exemptions?* Check your model card and §5.1 reflection text for stray references to the discarded labels.

### The defining teaching moments — narrowing AND OOD

**A model lives in a closed world bounded by its training, and you are responsible for both ends — the question it was trained on (§5.1) and the inputs it gets fed at deployment (§5.2).** This is the one thing the reader should remember when they walk away from this challenge.

### Extensions

1. **Replace the softmax-confidence OOD detector with a separate pre-classifier** that asks *is this a chest X-ray?* using a small CNN or even a hand-engineered input check (image dimensions, intensity histogram). Combine with the binary pneumonia classifier as a two-stage pipeline.
2. **Domain-shift evaluation.** Find a non-NIH chest X-ray dataset (e.g. CheXpert, OpenI) and report performance on it. Demo 2's Stanford story will repeat — internal AUC will exceed external AUC. Honest reporting is the contribution.
3. **Calibration.** Plot reliability diagrams (predicted probability vs. observed frequency) on the test set. The output of a fine-tuned ResNet is rarely calibrated by default; a temperature-scaling step after training brings predicted probabilities into closer alignment with true frequencies.
4. **Add a Whisper-transcribed dictation layer.** Take a brief radiologist audio dictation and append it to the model's structured output as a second source of clinical context for the GPT narrative. Closer to real-world deployment.

</details>

---

*Section 3 · Challenge 4 · OCAI Booklet · Spring 2026 Edition*